# Compute & store features for the whole dataset (HDF5)

**Goal:** using the provided face-detection and feature-extraction code, compute a feature matrix for
**every video** in the dataset and save it to disk as an **HDF5 file — one file per video**. Each file
holds a matrix of shape *(frames × features)*. Genuine and deepfake features are written to **separate
folders**, and the HDF5 tree mirrors the original video directory structure.

> **Notes on the provided code (adapted to this environment):**
> - Detection uses **MTCNN from `facenet-pytorch`** instead of the `mtcnn`/TensorFlow package; the
>   provided `eyes_angle` / `scaling_factor` / `crop_and_align` helpers are kept verbatim.
> - In `ssim`, the deprecated `multichannel=True` argument is replaced by `channel_axis=-1` (required by
>   modern scikit-image); the computation is identical.

## Step 1 — Paths to the example videos and to the dataset folders

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ.setdefault('OMP_NUM_THREADS', '1')

import glob
from pathlib import Path

root = Path.cwd()
while not (root / "Data").exists() and root != root.parent:
    root = root.parent

real_folder = root / "Data" / "VidTIMIT"                          # genuine videos
fake_folder = root / "Data" / "DeepfakeTIMIT" / "higher_quality"  # deepfake videos (high quality)
real_video  = str(real_folder / "fadg0" / "sa1.avi")
fake_video  = sorted(glob.glob(str(fake_folder / "fadg0" / "sa1-video-*.avi")))[0]
print("real folder:", real_folder)
print("fake folder:", fake_folder)

real folder: /Users/blake/AI Projects/Deepfake-Detection/Data/VidTIMIT
fake folder: /Users/blake/AI Projects/Deepfake-Detection/Data/DeepfakeTIMIT/higher_quality


## Step 2 — Provided face detection & cropping (adapted to facenet-pytorch)

In [2]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from facenet_pytorch import MTCNN

_detector = MTCNN(keep_all=False, device="cpu")

def eyes_angle(left_eye, right_eye):
    dY = right_eye[1] - left_eye[1]; dX = right_eye[0] - left_eye[0]
    return np.degrees(np.arctan2(dY, dX))

def scaling_factor(left_eye, right_eye, desired_left_eye, desired_right_eye):
    dY = right_eye[1] - left_eye[1]; dX = right_eye[0] - left_eye[0]
    dist = np.sqrt(dX**2 + dY**2)
    ddY = desired_right_eye[1] - desired_left_eye[1]; ddX = desired_right_eye[0] - desired_left_eye[0]
    return np.sqrt(ddX**2 + ddY**2) / dist

def crop_and_align(image, left_eye, right_eye, desired_image_width, desired_left_eye_percentage):
    angle = eyes_angle(left_eye, right_eye)
    desired_left_eye  = (desired_left_eye_percentage[0]*desired_image_width,
                         desired_left_eye_percentage[1]*desired_image_width)
    desired_right_eye = ((1.0-desired_left_eye_percentage[0])*desired_image_width,
                         desired_left_eye_percentage[1]*desired_image_width)
    scale = scaling_factor(left_eye, right_eye, desired_left_eye, desired_right_eye)
    eyes_center = ((left_eye[0]+right_eye[0])//2, (left_eye[1]+right_eye[1])//2)
    M = cv2.getRotationMatrix2D(eyes_center, angle, scale)
    M[0,2] += desired_image_width*0.5 - eyes_center[0]
    M[1,2] += desired_left_eye[1]     - eyes_center[1]
    w = h = desired_image_width
    return cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC)

def detect_face(image, desired_size=256, desired_left_eye_percentage=(0.35, 0.35)):
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    _, _, landmarks = _detector.detect(rgb, landmarks=True)
    if landmarks is None:
        return None
    left_eye  = tuple(float(v) for v in landmarks[0][0])
    right_eye = tuple(float(v) for v in landmarks[0][1])
    return crop_and_align(image, left_eye, right_eye, desired_size, desired_left_eye_percentage)

/Users/blake/opt/anaconda3/envs/deepfake-detect/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 3 — Provided feature extraction

For each face we compute **SSIM, normalized-RMSE and PSNR** between the face and a slightly blurred
copy of itself (these capture how much high-frequency detail the face has), concatenated with a
`num_hist_bins`-bin **intensity histogram**. `num_hist_bins` is a system parameter — more bins → more
features. Here the feature vector has length `3 + 32 = 35`.

In [3]:
import skimage.metrics

num_hist_bins = 32   # parameter: number of histogram bins (more bins -> more features)

def compute_hist(image, num_bins=32):
    hist, bins = np.histogram(image.ravel(), num_bins, [0, 255], density=True)
    return hist

def compute_blurred_image(image, kernel_size=3, sigma=0.5):
    return cv2.GaussianBlur(image, (kernel_size, kernel_size), sigma)

def mse(x, y):
    return skimage.metrics.normalized_root_mse(x, y)

def psnr(x, y):
    return skimage.metrics.peak_signal_noise_ratio(x, y, data_range=255)

def ssim(x, y):
    # provided code used multichannel=True; modern scikit-image uses channel_axis
    return skimage.metrics.structural_similarity(x, y, channel_axis=-1,
                                                 gaussian_weights=True, sigma=1.5,
                                                 use_sample_covariance=False, data_range=255)

def compute_features(image):
    image_blurred = compute_blurred_image(image)
    im_ssim = ssim(image, image_blurred)
    im_mse  = mse(image, image_blurred)
    im_psnr = psnr(image, image_blurred)
    im_hist = compute_hist(image, num_bins=num_hist_bins)
    return np.concatenate([[im_ssim], [im_mse], [im_psnr], im_hist])

NUM_FEATURES = 3 + num_hist_bins
print("Feature vector length:", NUM_FEATURES)

Feature vector length: 35


In [4]:
# Quick example: features for one real and one fake face
def first_face(video):
    cap = cv2.VideoCapture(video); cap.set(cv2.CAP_PROP_POS_FRAMES, 10)
    ok, frame = cap.read(); cap.release()
    return detect_face(frame, desired_size=256)

rf, ff = first_face(real_video), first_face(fake_video)
print("real feature (len %d):" % NUM_FEATURES, np.round(compute_features(rf)[:5], 4), "...")
print("fake feature (len %d):" % NUM_FEATURES, np.round(compute_features(ff)[:5], 4), "...")

[W NNPACK.cpp:64] Could not initialize NNPACK! Reason: Unsupported hardware.


real feature (len 35): [9.98200e-01 5.50000e-03 5.40079e+01 2.00000e-04 1.20000e-03] ...
fake feature (len 35): [9.99100e-01 4.20000e-03 5.63056e+01 3.00000e-04 1.50000e-03] ...


## Step 4 — Parameters & output layout

- **`FRAMES_PER_VIDEO`** — how many frames to sample per video (evenly spaced). This becomes the number
  of rows in each video's feature matrix.
- Output goes under **`Data/Features/`**, split into `real/` and `fake/` and mirroring the original
  folder tree, e.g. `Data/Features/real/fadg0/sa1.h5`.

In [5]:
import h5py

FRAMES_PER_VIDEO = 10
FEATURES_ROOT = root / "Data" / "Features"

def process_video(video_path, frames_per_video=FRAMES_PER_VIDEO, desired_size=256):
    """Return a (n_frames_with_face x NUM_FEATURES) feature matrix for one video."""
    cap = cv2.VideoCapture(str(video_path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if n <= 0:
        cap.release(); return np.empty((0, NUM_FEATURES))
    frame_ids = np.linspace(0, n - 1, min(frames_per_video, n)).astype(int)
    rows = []
    for fid in frame_ids:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(fid))
        ok, frame = cap.read()
        if not ok:
            continue
        try:
            face = detect_face(frame, desired_size=desired_size)
            if face is not None:
                rows.append(compute_features(face))
        except Exception:
            continue   # skip frames where detection/feature computation fails
    cap.release()
    return np.array(rows) if rows else np.empty((0, NUM_FEATURES))

def save_features_h5(out_path, matrix, label, video_path):
    out_path = Path(out_path); out_path.parent.mkdir(parents=True, exist_ok=True)
    with h5py.File(out_path, "w") as h:
        h.create_dataset("features", data=matrix, compression="gzip")
        h.attrs["label"]        = label                 # 'real' or 'fake'
        h.attrs["source_video"] = str(video_path)
        h.attrs["n_frames"]     = matrix.shape[0]
        h.attrs["n_features"]   = matrix.shape[1] if matrix.ndim == 2 else 0

def process_folder(video_folder, label):
    """Process every .avi under video_folder; write one HDF5 per video, mirroring the tree."""
    video_folder = Path(video_folder)
    out_root = FEATURES_ROOT / label
    videos = sorted(glob.glob(str(video_folder / "**" / "*.avi"), recursive=True))
    ok, empty = 0, 0
    for i, vp in enumerate(videos, 1):
        rel = Path(vp).relative_to(video_folder).with_suffix(".h5")
        out_path = out_root / rel
        matrix = process_video(vp)
        if matrix.shape[0] == 0:
            empty += 1
            continue
        save_features_h5(out_path, matrix, label, vp)
        ok += 1
        if i % 50 == 0 or i == len(videos):
            print(f"  [{label}] {i}/{len(videos)} videos processed")
    print(f"[{label}] done: {ok} HDF5 files written, {empty} videos with no detected face")
    return ok

## Step 5 — Run on the whole dataset

Genuine → `Data/Features/real/…`, deepfake → `Data/Features/fake/…` (this loop processes all 750
videos and takes several minutes).

In [6]:
n_real = process_folder(real_folder, "real")
n_fake = process_folder(fake_folder, "fake")
print(f"\nTotal: {n_real} real + {n_fake} fake HDF5 feature files -> {FEATURES_ROOT}")

  [real] 50/430 videos processed
  [real] 100/430 videos processed
  [real] 150/430 videos processed
  [real] 200/430 videos processed
  [real] 250/430 videos processed
  [real] 300/430 videos processed
  [real] 350/430 videos processed
  [real] 400/430 videos processed
  [real] 430/430 videos processed
[real] done: 430 HDF5 files written, 0 videos with no detected face
  [fake] 50/320 videos processed
  [fake] 100/320 videos processed
  [fake] 150/320 videos processed
  [fake] 200/320 videos processed
  [fake] 250/320 videos processed
  [fake] 300/320 videos processed
  [fake] 320/320 videos processed
[fake] done: 320 HDF5 files written, 0 videos with no detected face

Total: 430 real + 320 fake HDF5 feature files -> /Users/blake/AI Projects/Deepfake-Detection/Data/Features


## Step 6 — Verify the saved HDF5 files

In [7]:
real_h5 = sorted(glob.glob(str(FEATURES_ROOT / "real" / "**" / "*.h5"), recursive=True))
fake_h5 = sorted(glob.glob(str(FEATURES_ROOT / "fake" / "**" / "*.h5"), recursive=True))
print(f"HDF5 files -> real: {len(real_h5)}, fake: {len(fake_h5)}")

for path in (real_h5[0], fake_h5[0]):
    with h5py.File(path, "r") as h:
        feats = h["features"][:]
        rel = Path(path).relative_to(FEATURES_ROOT)
        print(f"\n{rel}")
        print(f"  matrix shape : {feats.shape}   (frames x features)")
        print(f"  label        : {h.attrs['label']}")
        print(f"  source_video : {Path(h.attrs['source_video']).name}")
        print(f"  first row[:5]: {np.round(feats[0][:5], 4)}")

HDF5 files -> real: 430, fake: 320

real/fadg0/sa1.h5
  matrix shape : (10, 35)   (frames x features)
  label        : real
  source_video : sa1.avi
  first row[:5]: [9.98200e-01 5.70000e-03 5.40623e+01 2.00000e-04 1.40000e-03]

fake/fadg0/sa1-video-fram1.h5
  matrix shape : (10, 35)   (frames x features)
  label        : fake
  source_video : sa1-video-fram1.avi
  first row[:5]: [9.99000e-01 4.90000e-03 5.55523e+01 3.00000e-04 2.40000e-03]


## Summary

- The provided detection + feature code was run over **every video** in the dataset.
- Each video produced a `(FRAMES_PER_VIDEO × 35)` feature matrix, saved as **one HDF5 file**.
- Files are split into `Data/Features/real/` and `Data/Features/fake/`, mirroring the original video
  directory tree, so each feature file's origin (genuine vs. deepfake) is unambiguous.

These HDF5 feature files are the input to the next step: **training a classifier** to tell real faces
from deepfakes.